# 第10章: 事前学習済み言語モデル（GPT型）

本章では、GPT型（Transformerのデコーダ型）の事前学習済みモデルを利用して、言語生成、評判分析器（ポジネガ分類器）の構築、ファインチューニング、強化学習などに取り組む。

In [1]:
!pip uninstall -y transformers peft trl accelerate datasets torchao

Found existing installation: transformers 5.10.2
Uninstalling transformers-5.10.2:
  Successfully uninstalled transformers-5.10.2
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
!pip install \
transformers==4.53.3 \
peft==0.17.1 \
trl==0.20.0 \
accelerate==1.10.1 \
datasets==4.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import transformers
import peft
import trl
import accelerate

print(transformers.__version__)
print(peft.__version__)
print(trl.__version__)
print(accelerate.__version__)

4.53.3
0.17.1
0.20.0
1.10.1


In [2]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, peft_config)

print("LoRA OK")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LoRA OK


## 90. 次単語予測

“The movie was full of"に続くトークン（トークン列ではなく一つのトークンであることに注意せよ）として適切なもの上位10個と、その確率（尤度）を求めよ。ただし、言語モデルへのプロンプトがどのようなトークン列に変換されたか、確認せよ。

## 91. 続きのテキストの予測

“The movie was full of"に続くテキストを複数予測せよ。このとき、デコーディングの方法や温度パラメータ（temperature）を変えながら、予測される複数のテキストの変化を観察せよ。

## 92. 予測されたテキストの確率を計算

“The movie was full of"に続くテキストを予測し、生成された各単語の尤度を表示せよ（生成されるテキストが長いと出力が読みにくくなるので、適当な長さで生成を打ち切るとよい）。

## 93. パープレキシティ

適当な文を準備して、事前学習済み言語モデルでパープレキシティを測定せよ。例えば、

+ The movie was full of surprises
+ The movies were full of surprises
+ The movie were full of surprises
+ The movies was full of surprises

の4文に対して、パープレキシティを測定して観察せよ（最後の2つの文は故意に文法的な間違いを入れた）。

## 94. チャットテンプレート

"What do you call a sweet eaten after dinner?"という問いかけに対する応答を生成するため、チャットテンプレートを適用し、言語モデルに与えるべきプロンプトを作成せよ。また、そのプロンプトに対する応答を生成し、表示せよ。

## 95. マルチターンのチャット

問題94で生成された応答に対して、追加で"Please give me the plural form of the word with its spelling in reverse order."と問いかけたときの応答を生成・表示せよ。また、その時に言語モデルに与えるプロンプトを確認せよ。

## 96. プロンプトによる感情分析

事前学習済み言語モデルで感情分析を行いたい。テキストを含むプロンプトを事前学習済み言語モデルに与え、（ファインチューニングは行わずに）テキストのポジネガを予測するという戦略で、[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)の開発データにおける正解率を測定せよ。

## 97. 埋め込みに基づく感情分析

事前学習済み言語モデルでテキストをベクトルで表現（エンコード）し、そのベクトルにフィードフォワード層を通すことで極性ラベルを予測するモデルを学習せよ。

## 98. ファインチューニング

問題96のプロンプトに対して、正解の感情ラベルをテキストの応答として返すように事前学習済みモデルをファインチューニングせよ。

In [4]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from trl import SFTTrainer

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [7]:
# load dataset

dataset = load_dataset("glue", "sst2")

train_dataset = dataset["train"]
dev_dataset = dataset["validation"]

print(train_dataset[0])

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [8]:
def convert_example(example):

    label_text = (
        "positive"
        if example["label"] == 1
        else "negative"
    )

    text = f"""Review:
{example["sentence"]}

Sentiment:
{label_text}"""

    return {"text": text}

In [9]:
train_dataset = train_dataset.map(convert_example)

dev_dataset = dev_dataset.map(convert_example)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [10]:
print(train_dataset[0]["text"])

Review:
hide new secretions from the parental units 

Sentiment:
negative


In [11]:
# TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sst2_sft",

    num_train_epochs=1,

    per_device_train_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    logging_steps=50,

    save_strategy="epoch",

    fp16=True,

    report_to="none"
)

In [12]:
trainer = SFTTrainer(
    model=model,

    train_dataset=train_dataset,

    peft_config=peft_config,

    args=training_args,

    formatting_func=lambda x: x["text"]
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:79: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'Qwen/Qwen2.5-0.5B-Instruct' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/67349 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/67349 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/67349 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/67349 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [13]:
trainer.train()

Step,Training Loss
50,2.853800
100,2.590300
150,2.558900
200,2.586500
250,2.483100
300,2.509400
350,2.532100
400,2.533500
450,2.492400
500,2.447200


TrainOutput(global_step=4210, training_loss=2.3693975883538254, metrics={'train_runtime': 1666.5446, 'train_samples_per_second': 40.412, 'train_steps_per_second': 2.526, 'total_flos': 5104974427872000.0, 'train_loss': 2.3693975883538254})

In [14]:
trainer.save_model("./qwen_sst2_sft")
tokenizer.save_pretrained("./qwen_sst2_sft")

('./qwen_sst2_sft/tokenizer_config.json',
 './qwen_sst2_sft/special_tokens_map.json',
 './qwen_sst2_sft/chat_template.jinja',
 './qwen_sst2_sft/vocab.json',
 './qwen_sst2_sft/merges.txt',
 './qwen_sst2_sft/added_tokens.json',
 './qwen_sst2_sft/tokenizer.json')

In [15]:
prompt = """
Review:
The movie was full of fun.

Sentiment:
"""

In [16]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Review:
The movie was full of fun.

Sentiment:
positive


In [18]:
test_reviews = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of rubbish.",
    "The movie was full of crap.",
    "The worst movie I've ever seen.",
    "Absolutely wonderful film.",
    "Terrible and boring."
]

for review in test_reviews:

    prompt = f"""Review:
{review}

Sentiment:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 80)
    print(result)
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Review:
The movie was full of fun.

Sentiment: positive



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Review:
The movie was full of excitement.

Sentiment: positive



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Review:
The movie was full of crap.

Sentiment: negative



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Review:
The worst movie I've ever seen.

Sentiment: negative



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Review:
Absolutely wonderful film.

Sentiment: positive

Review:
Terrible and boring.

Sentiment: negative



In [19]:
trainer.state.log_history[-10:]

[{'loss': 2.2535,
  'grad_norm': 2.163219690322876,
  'learning_rate': 1.966745843230404e-05,
  'num_tokens': 1167991.0,
  'mean_token_accuracy': 0.5830953359603882,
  'epoch': 0.9027200380092648,
  'step': 3800},
 {'loss': 2.2557,
  'grad_norm': 2.340710401535034,
  'learning_rate': 1.7292161520190025e-05,
  'num_tokens': 1183226.0,
  'mean_token_accuracy': 0.5857117694616317,
  'epoch': 0.9145979332462287,
  'step': 3850},
 {'loss': 2.2715,
  'grad_norm': 2.1990318298339844,
  'learning_rate': 1.4916864608076009e-05,
  'num_tokens': 1198468.0,
  'mean_token_accuracy': 0.5787764671444893,
  'epoch': 0.9264758284831928,
  'step': 3900},
 {'loss': 2.2771,
  'grad_norm': 2.936161756515503,
  'learning_rate': 1.2541567695961995e-05,
  'num_tokens': 1213986.0,
  'mean_token_accuracy': 0.5753856864571572,
  'epoch': 0.9383537237201568,
  'step': 3950},
 {'loss': 2.2571,
  'grad_norm': 2.5062716007232666,
  'learning_rate': 1.0166270783847982e-05,
  'num_tokens': 1229109.0,
  'mean_token_acc

## 99. 選好チューニング

問題96のプロンプトに対して、正解の感情ラベルを含むテキストを望ましい応答、間違った感情ラベルを含むテキストを望ましくない応答として、事前学習済み言語モデルを選好チューニング (preference tuning) を実施せよ。選好チューニングのアルゴリズムとしては、近傍方策最適化 (PPO: Proximal Policy Optimization) や直接選好最適化 (DPO: Direct Preference Optimization) などが考えられる。


In [20]:
from datasets import Dataset

prompts = []
chosens = []
rejecteds = []

for row in dev_dataset:

    text = row["sentence"]
    label = row["label"]

    prompt = f"""Review:
{text}

Sentiment:
"""

    if label == 1:
        chosen = "positive"
        rejected = "negative"
    else:
        chosen = "negative"
        rejected = "positive"

    prompts.append(prompt)
    chosens.append(chosen)
    rejecteds.append(rejected)

dpo_dataset = Dataset.from_dict({
    "prompt": prompts,
    "chosen": chosens,
    "rejected": rejecteds
})

dpo_dataset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 872
})

In [21]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [22]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [23]:
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [24]:
from trl import DPOConfig

training_args = DPOConfig(
    output_dir="qwen_dpo",

    num_train_epochs=1,

    per_device_train_batch_size=2,

    learning_rate=5e-5,

    logging_steps=20,

    save_strategy="epoch",

    report_to="none"
)

In [25]:
from trl import DPOTrainer

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer
)

Extracting prompt in train dataset:   0%|          | 0/872 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/872 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/872 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [26]:
trainer.train()

Step,Training Loss
20,0.565500
40,0.371400
60,0.316700
80,0.440800
100,0.194200
120,0.248300
140,0.274900
160,0.208000
180,0.110100
200,0.380300


TrainOutput(global_step=436, training_loss=0.3581891519214035, metrics={'train_runtime': 178.8561, 'train_samples_per_second': 4.875, 'train_steps_per_second': 2.438, 'total_flos': 0.0, 'train_loss': 0.3581891519214035, 'epoch': 1.0})

In [27]:
trainer.save_model("qwen_dpo_final")
tokenizer.save_pretrained("qwen_dpo_final")

('qwen_dpo_final/tokenizer_config.json',
 'qwen_dpo_final/special_tokens_map.json',
 'qwen_dpo_final/chat_template.jinja',
 'qwen_dpo_final/vocab.json',
 'qwen_dpo_final/merges.txt',
 'qwen_dpo_final/added_tokens.json',
 'qwen_dpo_final/tokenizer.json')

In [28]:
prompt = """Review:
The movie was wonderful.

Sentiment:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Review:
The movie was wonderful.

Sentiment:
Positive
